In [168]:
import pandas as pd
data=pd.read_csv("./csv/train.csv")
test=pd.read_csv("./csv/test.csv")
test_ids=test['PassengerId']
test.head(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [169]:
data.columns.tolist()
test.columns.tolist()

['PassengerId',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

In [170]:
def clean(hello):
    hello=hello.drop(['PassengerId','Name','Ticket','Cabin'],axis=1)
    cols=['Age','Fare','SibSp','Parch']
    for col in cols:
        hello[col]=hello[col].fillna(hello[col].mean())
    hello['Embarked'].fillna("U",inplace=True)
    return hello

data=clean(data)
test=clean(test)

/var/folders/yx/xyx8lw656pd4xjw4lhhqpny40000gn/T/ipykernel_5900/865099935.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  hello['Embarked'].fillna("U",inplace=True)
/var/folders/yx/xyx8lw656pd4xjw4lhhqpny40000gn/T/ipykernel_5900/865099935.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves

In [171]:
data.isnull().sum()
# test.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [172]:
from sklearn import preprocessing
le=preprocessing.LabelEncoder()

cols=['Sex','Embarked']

for col in cols:
    data[col] = le.fit_transform(data[col])
    test[col] = le.transform(test[col])
    print(le.classes_)

test.head(5)

['female' 'male']
['C' 'Q' 'S' 'U']


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,1,34.5,0,0,7.8292,1
1,3,0,47.0,1,0,7.0000,2
2,2,1,62.0,0,0,9.6875,1
3,3,1,27.0,0,0,8.6625,2
4,3,0,22.0,1,1,12.2875,2


In [173]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

y= data['Survived']
x=data.drop('Survived', axis=1)

x_train,X_val,y_train,y_val=train_test_split(x,y,test_size=0.2,random_state=42)

In [174]:
clf=LogisticRegression(random_state=0,max_iter=1000).fit(x_train,y_train)

/Users/rachit/Documents/Kaggle/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/rachit/Documents/Kaggle/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/rachit/Documents/Kaggle/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept


In [175]:
predictions = clf.predict(X_val)
from sklearn.metrics import accuracy_score
accuracy_score(y_val,predictions)

/Users/rachit/Documents/Kaggle/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rachit/Documents/Kaggle/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rachit/Documents/Kaggle/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


0.8100558659217877

In [176]:
submission_preds=clf.predict(test)

/Users/rachit/Documents/Kaggle/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rachit/Documents/Kaggle/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rachit/Documents/Kaggle/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [177]:
df1=pd.DataFrame({"PassengerId":test_ids.values,
                  "Survived":submission_preds})

In [178]:
df1.to_csv("./csv/gender_submission.csv",index=False)